# Exploration and NGU

Episodic novelty, intrinsic reward, and what the simplification omits.

> All examples run on CPU without model downloads and use synthetic data. They explain mechanisms; real training quality must be evaluated separately.

## What to understand

Never Give Up (2020) is an exploration algorithm, not a new GRPO variant. The original combines episodic novelty, lifelong novelty, policies with different intrinsic-reward weights, and off-policy recurrent learning.

Here we illustrate only an episodic bonus using distances to remembered states. This is neither full NGU nor its validation. Intrinsic reward is not task success: novelty can distract an agent. Reset episodic memory at episode boundaries.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Run the experiment

Run cells in order. Edit the code; Shift + Enter runs the current cell.

In [ ]:
states = torch.tensor([[0., 0.], [1., 0.], [0., 0.], [2., 0.], [1., 0.], [5., 0.]])
memory, bonuses = [], []
for state in states:
    if not memory:
        bonus = 1.0
    else:
        distances = torch.cdist(state[None], torch.stack(memory)).square()[0]
        nearest = distances.topk(min(3, len(memory)), largest=False).values
        similarity = (0.1 / (nearest + 0.1)).sum()
        bonus = float(1 / torch.sqrt(1 + similarity))
    bonuses.append(bonus); memory.append(state)
task_reward = torch.tensor([0., 0., 0., 0., 0., 1.])
mixed = task_reward + 0.2 * torch.tensor(bonuses)
print('novelty:', bonuses); print('mixed reward:', mixed.tolist())
plt.plot(bonuses, color='#d73535', marker='o'); plt.ylabel('illustrative novelty bonus'); plt.show()

## Try it yourself

Repeat one state 20 times, then add a distant one. Why should novelty use a learned representation instead of raw pixels?

Save a notebook copy and record what you changed, expected, and measured.

## Continue in Studio

The workspace link above the lesson opens related tools. This notebook does not launch large training jobs. Before real training, check data, resources, and method support for the chosen model.

## Sources

- [Never Give Up](https://arxiv.org/abs/2002.06038)